In [0]:
# Databricks notebook source

# 03 — Train and Register Official Price Model
#
This notebook:

1. Loads eligible Gold market observations.
2. Recreates the approved game-level data split.
3. Trains the official Ridge model on train + validation games.
4. Evaluates the model on the held-out test games.
5. Logs the approved decision thresholds.
6. Registers the model in Unity Catalog.
7. Reloads the registered model for a smoke test.

This notebook does not perform new model selection.

In [0]:
# Add the repository root to the Python path

import sys
from pathlib import Path


def find_repository_root(
    start_path: Path,
) -> Path:
    current_path = start_path.resolve()

    for candidate in [
        current_path,
        *current_path.parents,
    ]:
        if (
            candidate
            / "src"
            / "modeling"
        ).is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the repository root "
        "containing src/modeling."
    )


REPOSITORY_ROOT = find_repository_root(
    Path.cwd()
)

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(REPOSITORY_ROOT),
    )

print(
    f"Repository root: {REPOSITORY_ROOT}"
)

In [0]:
# Imports and registration settings

import json

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupShuffleSplit,
)

from src.modeling.evaluate import (
    evaluate_model,
)
from src.modeling.model_config import (
    GOLD_TABLE,
    FEATURE_COLUMNS,
    EVALUATION_COLUMNS,
    ELIGIBILITY_COLUMN,
    TARGET_COLUMN,
    GROUP_COLUMN,
    TRAIN_FRACTION,
    VALIDATION_FRACTION,
    TEST_FRACTION,
    RANDOM_STATE,
    MODEL_SELECTION_ARTIFACT_PATH,
    REGISTERED_MODEL_NAME,
    TARGET_HORIZON_MINUTES,
)
from src.modeling.register import (
    load_registered_model,
    register_model,
)
from src.modeling.train import (
    train_model,
)


print(
    f"Model-selection artifact: "
    f"{MODEL_SELECTION_ARTIFACT_PATH}"
)

print(
    f"Registered model name: "
    f"{REGISTERED_MODEL_NAME}"
)

In [0]:
# Load the model-selection artifact

artifact_path = Path(
    MODEL_SELECTION_ARTIFACT_PATH
)

if not artifact_path.exists():
    raise FileNotFoundError(
        "The model-selection artifact was not found. "
        "Run notebook 02 through Cell 33 first. "
        f"Expected path: {MODEL_SELECTION_ARTIFACT_PATH}"
    )


with artifact_path.open(
    mode="r",
    encoding="utf-8",
) as artifact_file:
    model_selection_artifact = (
        json.load(artifact_file)
    )


selected_model_name = (
    model_selection_artifact[
        "selected_model_name"
    ]
)

approved_thresholds = (
    model_selection_artifact[
        "validation_derived_thresholds"
    ]
)

artifact_feature_columns = (
    model_selection_artifact[
        "feature_columns"
    ]
)

artifact_target_column = (
    model_selection_artifact[
        "target_column"
    ]
)


print(
    f"Selected model: "
    f"{selected_model_name}"
)

print(
    f"Target column: "
    f"{artifact_target_column}"
)

print(
    f"Feature count: "
    f"{len(artifact_feature_columns)}"
)

print(
    f"Approved thresholds: "
    f"{approved_thresholds}"
)

In [0]:
# Validate the artifact against current configuration

if artifact_target_column != TARGET_COLUMN:
    raise ValueError(
        "The artifact target does not match "
        "the current model configuration. "
        f"Artifact: {artifact_target_column}. "
        f"Config: {TARGET_COLUMN}."
    )


if artifact_feature_columns != FEATURE_COLUMNS:
    raise ValueError(
        "The artifact feature order does not match "
        "FEATURE_COLUMNS in model_config.py. "
        "Feature order must remain unchanged."
    )


artifact_split_configuration = (
    model_selection_artifact[
        "split_configuration"
    ]
)

expected_split_configuration = {
    "train_fraction": float(
        TRAIN_FRACTION
    ),
    "validation_fraction": float(
        VALIDATION_FRACTION
    ),
    "test_fraction": float(
        TEST_FRACTION
    ),
    "random_state": int(
        RANDOM_STATE
    ),
}

if (
    artifact_split_configuration
    != expected_split_configuration
):
    raise ValueError(
        "The artifact split configuration does not "
        "match the current model configuration.\n"
        f"Artifact: {artifact_split_configuration}\n"
        f"Current: {expected_split_configuration}"
    )


required_threshold_keys = {
    "0.95",
    "0.975",
    "0.99",
}

missing_threshold_keys = (
    required_threshold_keys
    - set(approved_thresholds)
)

if missing_threshold_keys:
    raise ValueError(
        "The model-selection artifact is missing "
        "required opportunity thresholds. "
        f"Missing: {sorted(missing_threshold_keys)}"
    )


print(
    "Model-selection artifact passed "
    "configuration validation."
)

In [0]:
# Load eligible Gold observations

from pyspark.sql import functions as F


required_columns = list(
    dict.fromkeys(
        [
            GROUP_COLUMN,
            "kalshi_ticker",
            "minute_ts",
        ]
        + FEATURE_COLUMNS
        + EVALUATION_COLUMNS
        + [
            TARGET_COLUMN,
            ELIGIBILITY_COLUMN,
        ]
    )
)


modeling_spark_df = (
    spark.read.table(GOLD_TABLE)
    .select(*required_columns)
    .filter(
        F.col(ELIGIBILITY_COLUMN)
        == F.lit(True)
    )
    .dropna(
        subset=(
            FEATURE_COLUMNS
            + [
                TARGET_COLUMN,
                GROUP_COLUMN,
                "kalshi_ticker",
                "minute_ts",
            ]
        )
    )
    .orderBy(
        GROUP_COLUMN,
        "kalshi_ticker",
        "minute_ts",
    )
)


modeling_pdf = (
    modeling_spark_df.toPandas()
)

modeling_pdf["minute_ts"] = (
    pd.to_datetime(
        modeling_pdf["minute_ts"],
        utc=True,
    )
)


print(
    f"Eligible observations: "
    f"{len(modeling_pdf):,}"
)

print(
    f"Independent games: "
    f"{modeling_pdf[GROUP_COLUMN].nunique():,}"
)

print(
    f"Kalshi tickers: "
    f"{modeling_pdf['kalshi_ticker'].nunique():,}"
)

In [0]:
# Recreate the development and test split

split_fraction_sum = (
    TRAIN_FRACTION
    + VALIDATION_FRACTION
    + TEST_FRACTION
)

if not np.isclose(
    split_fraction_sum,
    1.0,
):
    raise ValueError(
        "Train, validation, and test fractions "
        "must sum to 1.0. "
        f"Current sum: {split_fraction_sum}"
    )


development_fraction = (
    TRAIN_FRACTION
    + VALIDATION_FRACTION
)


outer_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=development_fraction,
    random_state=RANDOM_STATE,
)


development_indices, test_indices = next(
    outer_splitter.split(
        modeling_pdf,
        groups=modeling_pdf[
            GROUP_COLUMN
        ],
    )
)


development_pdf = (
    modeling_pdf
    .iloc[development_indices]
    .copy()
    .reset_index(drop=True)
)

test_pdf = (
    modeling_pdf
    .iloc[test_indices]
    .copy()
    .reset_index(drop=True)
)


development_games = set(
    development_pdf[GROUP_COLUMN]
)

test_games = set(
    test_pdf[GROUP_COLUMN]
)


overlapping_games = (
    development_games
    .intersection(test_games)
)

if overlapping_games:
    raise ValueError(
        "Game leakage detected between "
        "development and test data."
    )


print(
    f"Development observations: "
    f"{len(development_pdf):,}"
)

print(
    f"Development games: "
    f"{len(development_games):,}"
)

print(
    f"Test observations: "
    f"{len(test_pdf):,}"
)

print(
    f"Test games: "
    f"{len(test_games):,}"
)

In [0]:
# Confirm the selected official model

print(
    f"Selected official model: "
    f"{selected_model_name}"
)

In [0]:
# Train the official model

X_development = (
    development_pdf[
        FEATURE_COLUMNS
    ]
)

y_development = (
    development_pdf[
        TARGET_COLUMN
    ]
)

X_test = (
    test_pdf[
        FEATURE_COLUMNS
    ]
)

y_test = (
    test_pdf[
        TARGET_COLUMN
    ]
)


official_model = train_model(
    model_name=selected_model_name,
    X_train=X_development,
    y_train=y_development,
)


print(
    f"Trained {selected_model_name} on "
    f"{len(development_pdf):,} observations "
    f"from {len(development_games):,} games."
)

In [0]:
# Evaluate the model on test games

test_predictions, raw_test_metrics = (
    evaluate_model(
        model=official_model,
        X=X_test,
        y=y_test,
    )
)


test_metrics = {
    "test_mae": float(
        raw_test_metrics["mae"]
    ),
    "test_rmse": float(
        raw_test_metrics["rmse"]
    ),
    "test_r2": float(
        raw_test_metrics["r2"]
    ),
    "test_directional_accuracy_nonzero": float(
        raw_test_metrics[
            "directional_accuracy_nonzero"
        ]
    ),
}


test_metrics_pdf = pd.DataFrame(
    [
        {
            "model_name": (
                selected_model_name
            ),
            **test_metrics,
        }
    ]
)


display(test_metrics_pdf)

In [0]:
# Prepare registration inputs

if X_development.empty:
    raise ValueError(
        "X_development is empty."
    )

if len(FEATURE_COLUMNS) == 0:
    raise ValueError(
        "FEATURE_COLUMNS is empty."
    )


missing_input_columns = [
    column_name
    for column_name in FEATURE_COLUMNS
    if column_name
    not in X_development.columns
]

if missing_input_columns:
    raise ValueError(
        "X_development is missing required "
        f"features: {missing_input_columns}"
    )


input_example = (
    X_development[
        FEATURE_COLUMNS
    ]
    .head(5)
    .copy()
)

if input_example.empty:
    raise ValueError(
        "The MLflow input example "
        "contains no rows."
    )


registration_params = {
    "model_type": (
        selected_model_name
    ),
    "target_column": (
        TARGET_COLUMN
    ),
    "target_horizon_minutes": (
        TARGET_HORIZON_MINUTES
    ),
    "feature_count": (
        len(FEATURE_COLUMNS)
    ),
    "development_row_count": (
        len(development_pdf)
    ),
    "development_game_count": (
        len(development_games)
    ),
    "test_row_count": (
        len(test_pdf)
    ),
    "test_game_count": (
        len(test_games)
    ),
    "random_state": (
        RANDOM_STATE
    ),
}


registration_metadata = {
    "selected_model_name": (
        selected_model_name
    ),
    "feature_columns": list(
        FEATURE_COLUMNS
    ),
    "target_column": (
        TARGET_COLUMN
    ),
    "evaluation_columns": list(
        EVALUATION_COLUMNS
    ),
    "eligibility_column": (
        ELIGIBILITY_COLUMN
    ),
}


print(
    f"Input example rows: "
    f"{len(input_example)}"
)

print(
    f"Input example columns: "
    f"{list(input_example.columns)}"
)

print(
    input_example.to_string(
        index=False
    )
)

In [0]:
# Log and register the model

model_version, run_id = (
    register_model(
        model=official_model,
        input_example=input_example,
        metrics=test_metrics,
        params=registration_params,
        metadata=registration_metadata,
        decision_thresholds=(
            approved_thresholds
        ),
        model_selection_artifact=(
            model_selection_artifact
        ),
        run_name=(
            "official_kalshi_cfb_"
            f"{selected_model_name}"
        ),
    )
)


print(
    f"MLflow run ID: "
    f"{run_id}"
)

print(
    f"Registered model: "
    f"{REGISTERED_MODEL_NAME}"
)

print(
    f"Registered version: "
    f"{model_version}"
)

In [0]:
# Reload the registered model version

loaded_model, registered_model_uri = (
    load_registered_model(
        model_version=model_version
    )
)


print(
    f"Loaded registered model from: "
    f"{registered_model_uri}"
)

In [0]:
# Confirm reloaded predictions match

original_smoke_predictions = (
    official_model.predict(
        X_test.head(100)
    )
)

reloaded_smoke_predictions = (
    loaded_model.predict(
        X_test.head(100)
    )
)


maximum_prediction_difference = (
    np.max(
        np.abs(
            original_smoke_predictions
            - reloaded_smoke_predictions
        )
    )
)


if not np.allclose(
    original_smoke_predictions,
    reloaded_smoke_predictions,
):
    raise ValueError(
        "Predictions from the registered model "
        "do not match the in-memory model."
    )


print(
    "Registered-model prediction check passed."
)

print(
    f"Maximum prediction difference: "
    f"{maximum_prediction_difference:.12f}"
)

In [0]:
# Score a sample and assign signal tiers


def assign_signal_tier(
    prediction: float,
    thresholds: dict[str, float],
) -> int:
    if prediction >= float(
        thresholds["0.99"]
    ):
        return 3

    if prediction >= float(
        thresholds["0.975"]
    ):
        return 2

    if prediction >= float(
        thresholds["0.95"]
    ):
        return 1

    return 0


smoke_test_pdf = (
    test_pdf
    .head(1000)
    .copy()
)


smoke_test_pdf[
    "predicted_price_change_5m"
] = loaded_model.predict(
    smoke_test_pdf[
        FEATURE_COLUMNS
    ]
)


smoke_test_pdf[
    "signal_tier"
] = (
    smoke_test_pdf[
        "predicted_price_change_5m"
    ]
    .apply(
        lambda prediction: (
            assign_signal_tier(
                prediction=prediction,
                thresholds=(
                    approved_thresholds
                ),
            )
        )
    )
)


display(
    smoke_test_pdf[
        [
            GROUP_COLUMN,
            "kalshi_ticker",
            "minute_ts",
            "current_price",
            "predicted_price_change_5m",
            "signal_tier",
        ]
    ]
    .sort_values(
        by=(
            "predicted_price_change_5m"
        ),
        ascending=False,
    )
)

In [0]:
# Show tier counts

tier_summary_pdf = (
    smoke_test_pdf
    .groupby(
        "signal_tier",
        as_index=False,
    )
    .agg(
        observation_count=(
            "kalshi_ticker",
            "size",
        ),
        mean_prediction=(
            "predicted_price_change_5m",
            "mean",
        ),
        maximum_prediction=(
            "predicted_price_change_5m",
            "max",
        ),
    )
    .sort_values(
        by="signal_tier"
    )
    .reset_index(drop=True)
)

display(tier_summary_pdf)

In [0]:
# Final summary

print("Official model registration complete")
print("====================================")

print(
    f"Selected model: "
    f"{selected_model_name}"
)

print(
    f"Registered model: "
    f"{REGISTERED_MODEL_NAME}"
)

print(
    f"Registered version: "
    f"{model_version}"
)

print(
    f"MLflow run ID: "
    f"{run_id}"
)

print(
    f"Target: "
    f"{TARGET_COLUMN}"
)

print(
    f"Feature count: "
    f"{len(FEATURE_COLUMNS)}"
)

print()
print("Approved decision thresholds")
print("----------------------------")

for percentile in [
    "0.95",
    "0.975",
    "0.99",
]:
    print(
        f"{percentile}: "
        f"{float(approved_thresholds[percentile]):.6f}"
    )

print()
print("Test metrics")
print("------------")

for metric_name, metric_value in (
    test_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:.6f}"
    )